# Enterprise RAG Assistant
A complete Colab notebook for PDF Question Answering using FAISS + SentenceTransformers + Groq.

In [20]:
!pip -q install pymupdf sentence-transformers faiss-cpu langchain-text-splitters groq

In [21]:
import fitz, faiss, numpy as np
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import files
from groq import Groq

## Upload PDFs

In [29]:
uploaded = files.upload()

Saving Ankit-Resume.pdf to Ankit-Resume (3).pdf


In [30]:
GROQ_API_KEY=Grok.env('GROQ_API_KEY')
client=Groq(api_key=GROQ_API_KEY)
model=SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Helper Functions

In [31]:
def extract_documents(uploaded):

    docs=[]
    for name in uploaded.keys():
        pdf=fitz.open(name)
        for i,page in enumerate(pdf):
            docs.append({'file':name,'page':i+1,'text':page.get_text()})
    return docs

In [32]:
def chunk_documents(docs):
    splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
    chunks=[]
    for d in docs:
        for c in splitter.split_text(d['text']):
            chunks.append({'text':c,'file':d['file'],'page':d['page']})
    return chunks


In [33]:
def build_index(chunks):
    embs=model.encode([c['text'] for c in chunks],convert_to_numpy=True)
    index=faiss.IndexFlatL2(embs.shape[1]); index.add(embs.astype('float32'))
    return index,embs

In [34]:
docs=extract_documents(uploaded)
chunks=chunk_documents(docs)
index,embs=build_index(chunks)
print('Chunks:',len(chunks))

Chunks: 12


In [36]:
def answer(query,k=4):
    q=model.encode([query],convert_to_numpy=True).astype('float32')
    D,I=index.search(q,k)
    context=''
    sources=[]
    for idx in I[0]:
        c=chunks[idx]
        context+=c['text']+'\n\n'
        sources.append(f"{c['file']} (Page {c['page']})")
    prompt=f'''You are a helpful assistant. Answer ONLY from the context.
If unavailable, say you couldn't find it.

Context:
{context}

Question: {query}
Answer:'''
    resp=client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role':'user','content':prompt}],
        temperature=0)
    print(resp.choices[0].message.content)
    print("\nSources:")
    for s in sorted(set(sources)):
        print('-',s)

while True:
    q=input("Ask a question (type exit to stop): ")
    if q.lower()=='exit':
        break
    answer(q)

Ask a question (type exit to stop): What is this Document about?
This document appears to be a resume or a portfolio, highlighting the author's technical skills, projects, academic achievements, and experiences.

Sources:
- Ankit-Resume (3).pdf (Page 1)
Ask a question (type exit to stop): What is the resume specific?
The resume is specific to Ankit Saluja, a Final Year Postgraduate Student in the Department of Mathematics and Statistics at IIT Kanpur.

Sources:
- Ankit-Resume (3).pdf (Page 1)
Ask a question (type exit to stop): exit
